# 🎙️ EduPod TTS Server - Chatterbox TURBO on Colab T4

**⚠️ IMPORTANT: Run each cell ONE BY ONE!**

Chatterbox Turbo is the fastest & highest quality model from ResembleAI.

In [ ]:
# Cell 1: Fix NumPy (REQUIRES RUNTIME RESTART AFTER)
!pip uninstall numpy -y
!pip install 'numpy==1.24.4'
!apt-get update -qq && apt-get install -qq -y ffmpeg libsndfile1

print("\n" + "="*50)
print("⚠️ NOW: Go to Runtime -> Restart runtime")
print("   Then run Cell 2")
print("="*50)

In [ ]:
# Cell 2: Install Dependencies (AFTER RESTART)
import numpy as np
print(f"NumPy: {np.__version__}")
assert np.__version__.startswith('1.24'), "Wrong NumPy! Restart runtime first!"

!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate safetensors huggingface_hub diffusers
!pip install soundfile librosa scipy
!pip install conformer einops g2p-en unidecode phonemizer
!pip install s3tokenizer resemble-perth
!pip install fastapi uvicorn pyngrok nest-asyncio
!pip install chatterbox-tts --no-deps

print("\n✅ All dependencies installed!")

In [ ]:
# Cell 3: Test Imports
import torch, numpy as np
print(f"PyTorch: {torch.__version__}, NumPy: {np.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from chatterbox.tts_turbo import ChatterboxTurboTTS
print("\n✅ All imports OK!")

In [ ]:
# Cell 4: START EVERYTHING (Model + Server + ngrok)
# ================================================
# This is the ONLY cell you need to run after Cell 3!
# Set your ngrok token below:

NGROK_TOKEN = "YOUR_TOKEN"  # <-- REPLACE WITH YOUR TOKEN FROM https://dashboard.ngrok.com

# ========== DO NOT EDIT BELOW THIS LINE ==========

import torch
import torchaudio as ta
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import FileResponse
from pydantic import BaseModel
from typing import List
import tempfile, uuid, os
import nest_asyncio, uvicorn
from pyngrok import ngrok
from chatterbox.tts_turbo import ChatterboxTurboTTS

# Create the app
app = FastAPI(title="EduPod Chatterbox Turbo API")
AUDIO_DIR = tempfile.mkdtemp()

# Load model and store in app.state (GUARANTEED to be accessible)
print("🔊 Loading Chatterbox TURBO model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
app.state.tts_model = ChatterboxTurboTTS.from_pretrained(device=device)
app.state.sample_rate = app.state.tts_model.sr
print(f"✅ Model loaded on {device}! Sample Rate: {app.state.sample_rate}")

class DialogueRequest(BaseModel):
    lines: List[dict]

@app.get("/")
def health(request: Request):
    return {
        "status": "ok", 
        "model": "chatterbox-turbo", 
        "device": device,
        "model_loaded": hasattr(request.app.state, 'tts_model')
    }

@app.post("/synthesize_dialogue")
async def synthesize(req: DialogueRequest, request: Request):
    try:
        # Get model from app.state (ALWAYS works!)
        tts = request.app.state.tts_model
        sr = request.app.state.sample_rate
        
        wavs, meta = [], []
        for i, line in enumerate(req.lines):
            h = line.get("host", "host_1")
            c = line.get("content", "").strip()
            if not c: 
                continue
            
            print(f"  📝 Generating line {i+1}: {c[:50]}...")
            wav = tts.generate(c)
            wavs.append(wav)
            meta.append({"host": h, "content": c, "duration": wav.shape[-1]/sr})
            
            # Add 0.5s pause between lines
            wavs.append(torch.zeros(1, int(0.5 * sr)))
        
        if wavs:
            combined = torch.cat(wavs, dim=-1)
            fn = f"{uuid.uuid4()}.wav"
            fpath = os.path.join(AUDIO_DIR, fn)
            ta.save(fpath, combined, sr)
            print(f"  ✅ Saved: {fn}")
            return {"audio_url": f"/audio/{fn}", "metadata": meta}
        
        raise HTTPException(400, "No valid lines to synthesize")
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(500, str(e))

@app.get("/audio/{fn}")
async def get_audio(fn: str):
    p = os.path.join(AUDIO_DIR, fn)
    if os.path.exists(p):
        return FileResponse(p, media_type="audio/wav")
    raise HTTPException(404, "Audio file not found")

# Start server
nest_asyncio.apply()
if NGROK_TOKEN != "YOUR_TOKEN":
    ngrok.set_auth_token(NGROK_TOKEN)

url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"🚀 CHATTERBOX TURBO API: {url}")
print(f"\n📋 Set this in your environment:")
print(f"   set CHATTERBOX_API_URL={url}")
print("="*60 + "\n")

uvicorn.run(app, host="0.0.0.0", port=8000)